# Real-World Solution for SCD Type - 1 Dim

Initial Load/Full Load + Incremental Load

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
init_load_flag = dbutils.widgets.text('init_load_flag', '1')

print(init_load_flag)

# Data Reading

In [0]:
df = spark.read.table('medallion_arc_e2e.silver.customers')

display(df)

# Removing Duplicates

In [0]:
df = df.dropDuplicates(subset=['customer_id'])

display(df.limit(10))

# Dividing New vs Old Records

In [0]:
if init_load_flag == 0:

    df_old = spark.sql('''select dim_customer_key, customer_id, create_date, update_date
                       from medallion_arc_e2e.gold.dim_customers''')
else:

    df_old = spark.sql('''select dim_customer_key, customer_id, create_date, update_date
                       from medallion_arc_e2e.gold.dim_customers where 1=0''')

# Surrogate Key - All The Values

In [0]:
df = df.withColumn('dim_customer_key', monotonically_increasing_id()+lit(1))

display(df.limit(20))